In [ ]:
from math import sqrt
from numpy import zeros, set_printoptions, genfromtxt, stack, savetxt
from matplotlib import pyplot as plt
import matplotlib
matplotlib.rcParams['axes.formatter.useoffset'] = False
matplotlib.rcParams['axes.formatter.limits'] = (-5, 8)
from pyproj import Transformer

set_printoptions(precision=3, suppress=True)  # print altid 3 decimaler når arrays udskrives.

## Indlæs data fra en GNSS RTK fil

In [4]:
raw_data = genfromtxt('C:\\Users\\saschu\\OneDrive - Danmarks Tekniske Universitet\\Undervisning\\Site Investigations\\Site Investigations 2025\\krishp.txt', skip_header=1, delimiter=",", usecols=(2,1,3))  #,
print(raw_data[0:6])  # Vi printer lige et par punkter for at se hvordan de ser ud

[[ 383701.138 7427198.751      87.883]
 [ 383702.876 7427197.002      87.763]
 [ 383704.571 7427195.228      87.474]
 [ 383705.996 7427193.227      87.296]
 [ 383707.52  7427191.224      87.191]
 [ 383708.793 7427189.16       87.067]]


## transformer højderne

Vi bruger oftest højde over havniveau og ikke ellipsoide højder, vi skal derfor først transformere vores data med PROJ.
Vi laver derfor en pipeline der:
  - transformerer fra UTM zone 22 til geografisk
  - transformerer fra ellipsoidehøjder til højde over havniveau
  - transformerer fra geografisk tilbage til UTM zone 22

In [ ]:
pipeline = "+ellps=GRS80 +proj=pipeline +step +inv +proj=utm +zone=22 +step +proj=vgridshift +grids=dk_sdfe_gvr2016.tif +step +proj=utm +zone=22"
transform_object = Transformer.from_pipeline(pipeline)

raw_trans_data = stack(transform_object.transform((raw_data[:, 0]), (raw_data[:, 1]), (raw_data[:, 2])), axis=1)

print(raw_trans_data[0:6])  # Vi printer lige et par punkter for at se hvordan de ser ud

## Gem de "gode" data
I eksemplet er første række og sidste række ikke en del af profilen men nulpunkter for profilen, så dem gemmer vi i variabeler og skærer væk fra profilen.

Du kan fjerne flere rækker hvis du fik startet RTK'en for tidligt eller stoppet den for sent.

In [ ]:
zero_south = raw_trans_data[0]
print(f"Nulpunkt: {zero_south}")

profil_enu = raw_trans_data[1:-1]  # <-- Her springer vi det første punkt over (derfor 1 før kolon) og det sidste punkt springes også over (derfor -1 efter kolon)
print("Profil observationer:")
print(profil_enu)

## Vi kan nu beregne afstand langs profilen

$$l = \sqrt{(E - E_0)^2 + (N - N_0)^2}$$

In [ ]:
profil_lh = zeros((len(profil_enu), 2))

for (i, obs) in enumerate(profil_enu):
    profil_lh[i, 0] = sqrt((obs[0] - zero_south[0]) ** 2 + (obs[1] - zero_south[1]) ** 2)
    profil_lh[i, 1] = obs[2]
    
print(profil_lh[0:3])  # Vi printer lige et par punkter for at se hvordan de ser ud

In [ ]:
plt.scatter(profil_lh[:, 0], profil_lh[:, 1], color='tab:blue',label=f'Højdeprofil')

plt.xlabel('Afstand')
plt.ylabel('Højde')
plt.legend()
plt.grid(True)
plt.show()

## Gem data

Til sidst skal I gemme jeres data i en fil til senere brug.

In [ ]:
savetxt("vejprofil.csv", profil_lh, delimiter=",")